# 06 · Interim report: data cleaning, EDA and baseline models

Run **after notebook 01**. Produces every table and figure for the interim report and saves them to
`MyDrive/audio-detection-human-vs-ai/reports_interim/`. The last cell downloads them as one zip.

Runtime on Colab: about 60–90 min, mostly the 64-feature extraction (cached, so a re-run is fast).
Use **Runtime → Run all** and leave the tab open.

In [ ]:
# --- Setup: works on Google Colab and locally -------------------------------
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ.setdefault("VS_DATA_ROOT", "/content/drive/MyDrive/audio-detection-human-vs-ai")
    if not os.path.exists("/content/audio-detection-human-vs-ai"):
        !git clone -q https://github.com/abhilash-u/audio-detection-human-vs-ai /content/audio-detection-human-vs-ai
    %cd /content/audio-detection-human-vs-ai
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import time, hashlib, json, warnings
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from src import config as C
warnings.filterwarnings("ignore")
C.ensure_dirs()
OUT = C.DATA_ROOT / "reports_interim"          # saved in Google Drive, survives the session
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"font.family": "serif", "font.size": 11, "axes.titlesize": 11, "axes.labelsize": 11,
                     "figure.dpi": 110, "savefig.dpi": 200, "axes.spines.top": False, "axes.spines.right": False})
HUMAN, SYN = "#2F4B66", "#C46A3A"
def save(fig, name):
    fig.savefig(OUT / f"{name}.png", bbox_inches="tight", facecolor="white"); plt.show()
def table(df, name):
    df.to_csv(OUT / f"{name}.csv", index=False); return df
print("Data folder:", C.DATA_ROOT, "| outputs:", OUT)

## 1. Load metadata from notebook 01

In [ ]:
meta = pd.read_csv(C.META_DIR / "audio_metadata.csv")
sample = pd.read_csv(C.META_DIR / "split_assignments.csv")
print("Full database:", meta.shape, "| Study sample:", sample.shape)
meta["label_name"] = meta.label.map({0: "human", 1: "synthetic"})
sample["label_name"] = sample.label.map({0: "human", 1: "synthetic"})
overview = (meta.groupby(["official_partition", "label_name"]).agg(clips=("clip_id", "size"), speakers=("speaker_id", "nunique"))
            .reset_index())
table(overview, "T01_database_overview")

In [ ]:
speakers = meta.groupby("official_partition").speaker_id.nunique().rename("speakers").reset_index()
table(speakers, "T02_speakers_per_partition")

## 2. Data cleaning log: missing values, duplicates, unreadable files, outliers

In [ ]:
from src.preprocess import load
log = []
# Missing values in the metadata (attack_system_id is blank by design for human clips)
miss = meta.isna().sum()
miss_tbl = pd.DataFrame({"variable": miss.index, "missing": miss.values, "pct": (100 * miss.values / len(meta)).round(3)})
table(miss_tbl, "T03_missing_values_metadata")
expected_blank = int((meta.label == 0).sum())
log.append({"issue": "Missing values", "variables": "all metadata fields",
            "detection": "isna() count per column",
            "found": f"{int(miss.sum())} missing cells across {int((miss > 0).sum())} columns",
            "treatment": "attack_system_id / attack_type / system_status blank for human clips by design; no imputation needed"})
# Duplicate clip identifiers
dup_ids = int(meta.clip_id.duplicated().sum())
log.append({"issue": "Duplicate records", "variables": "clip_id", "detection": "duplicated() on clip_id",
            "found": f"{dup_ids} duplicate IDs", "treatment": "none needed" if dup_ids == 0 else "dropped"})
miss_tbl[miss_tbl.missing > 0]

In [ ]:
# Duplicate audio content and unreadable files in the study sample (hash of processed audio)
def file_hash(cid):
    p = C.CLEAN_DIR / f"{cid}.wav"
    try:
        return hashlib.md5(p.read_bytes()).hexdigest()
    except Exception:
        return None
t0 = time.time()
sample["audio_md5"] = [file_hash(c) for c in sample.clip_id]
unreadable = int(sample.audio_md5.isna().sum())
dup_audio = int(sample.audio_md5.dropna().duplicated().sum())
print(f"hashed {len(sample)} files in {time.time() - t0:.0f}s | unreadable: {unreadable} | duplicate audio: {dup_audio}")
log.append({"issue": "Unreadable files", "variables": "audio file", "detection": "read every processed clip",
            "found": f"{unreadable} of {len(sample)}", "treatment": "excluded" if unreadable else "none needed"})
log.append({"issue": "Duplicate audio", "variables": "audio content", "detection": "MD5 hash of processed audio",
            "found": f"{dup_audio} duplicates ({100 * dup_audio / len(sample):.2f}%)",
            "treatment": "kept one copy" if dup_audio else "none needed"})

In [ ]:
# Duration outliers (IQR rule on raw duration) and the silence shortcut
q1, q3 = sample.duration_raw_s.quantile([0.25, 0.75]); iqr = q3 - q1
lo_b, hi_b = q1 - 1.5 * iqr, q3 + 1.5 * iqr
out_mask = (sample.duration_raw_s < lo_b) | (sample.duration_raw_s > hi_b)
n_out = int(out_mask.sum())
short = int((sample.duration_trimmed_s < 1.0).sum())
log.append({"issue": "Outliers (duration)", "variables": "duration_raw_s", "detection": f"IQR rule: outside [{lo_b:.2f}, {hi_b:.2f}] s",
            "found": f"{n_out} clips ({100 * n_out / len(sample):.1f}%)",
            "treatment": "kept; every clip is cropped or repeat-padded to 4 s, so length cannot drive the model"})
log.append({"issue": "Very short speech", "variables": "duration_trimmed_s", "detection": "< 1.0 s of speech after trimming",
            "found": f"{short} clips ({100 * short / len(sample):.1f}%)", "treatment": "kept; repeat-padded to 4 s"})
log.append({"issue": "Label leakage via silence", "variables": "leading/trailing silence", "detection": "silence-only classifier AUC (notebook 01)",
            "found": "see T05", "treatment": "silence trimmed at 30 dB below peak; verified AUC near 0.5 after processing"})
cleaning_log = table(pd.DataFrame(log), "T04_cleaning_log")
cleaning_log

In [ ]:
qc = pd.read_csv(C.REPORT_DIR / "qc_shortcut_check.csv") if (C.REPORT_DIR / "qc_shortcut_check.csv").exists() else None
if qc is None:  # recompute if notebook 01 ran in another session
    from src.preprocess import processed_silence
    from src.evaluation import shortcut_auc
    s2 = sample.merge(processed_silence(sample.clip_id.tolist()), on="clip_id")
    tr, va = s2[s2.split == "training"], s2[s2.split == "validation"]
    qc = pd.DataFrame([{"check": "silence shortcut",
                        "auc_raw": shortcut_auc(tr, va, ["duration_raw_s", "leading_silence_s", "trailing_silence_s"]),
                        "auc_processed": shortcut_auc(tr, va, ["proc_leading_silence_s", "proc_trailing_silence_s"])}])
table(qc, "T05_shortcut_check")

## 3. EDA on the sample: composition, durations, silence, speaker clustering

In [ ]:
comp = sample.groupby(["split", "label_name"]).size().unstack(fill_value=0)
comp = comp.reindex(["training", "validation", "test_human", "test_known", "test_unknown"])
table(comp.reset_index(), "T06_sample_composition")
fig, ax = plt.subplots(figsize=(7, 3.4))
comp.plot.barh(stacked=True, color=[HUMAN, SYN], ax=ax, width=0.6)
ax.invert_yaxis(); ax.set_xlabel("Number of clips"); ax.set_ylabel(""); ax.legend(title="", frameon=False)
for i, (h, s) in enumerate(comp.values):
    ax.text(h + s + 40, i, f"{h + s:,}", va="center", fontsize=9)
save(fig, "F01_sample_composition")

In [ ]:
desc = sample.groupby("label_name")[["duration_raw_s", "leading_silence_s", "trailing_silence_s", "duration_trimmed_s"]].describe().T.round(3)
desc = desc.reset_index().rename(columns={"level_0": "variable", "level_1": "statistic"})
table(desc, "T07_duration_silence_by_class")
fig, axes = plt.subplots(1, 3, figsize=(10, 3.2))
for ax, col, lab in zip(axes, ["duration_raw_s", "leading_silence_s", "duration_trimmed_s"],
                        ["Raw duration (s)", "Leading silence (s)", "Speech after trimming (s)"]):
    for name, colr in [("human", HUMAN), ("synthetic", SYN)]:
        v = sample.loc[sample.label_name == name, col]
        ax.hist(v.clip(upper=v.quantile(0.995)), bins=40, alpha=0.6, color=colr, label=name, density=True)
    ax.set_xlabel(lab); ax.set_ylabel("Density")
axes[0].legend(frameon=False)
save(fig, "F02_duration_silence_by_class")

In [ ]:
per_spk = sample.groupby(["split", "speaker_id"]).size().rename("clips").reset_index()
spk_tbl = per_spk.groupby("split").clips.agg(["count", "mean", "median", "min", "max"]).rename(columns={"count": "speakers"}).round(1)
table(spk_tbl.reset_index(), "T08_clips_per_speaker")
fig, ax = plt.subplots(figsize=(7, 3.2))
order = ["training", "validation", "test_human", "test_known", "test_unknown"]
ax.boxplot([per_spk.loc[per_spk.split == s, "clips"] for s in order], labels=order, vert=False)
ax.set_xlabel("Clips per speaker"); ax.invert_yaxis()
save(fig, "F03_clips_per_speaker")
spk_tbl

## 4. Extract the 64 acoustic features (cached in Drive)

In [ ]:
from src import features as F
feat_path = C.FEAT_DIR / "features_clean.csv"
if feat_path.exists():
    X = pd.read_csv(feat_path)
    print("Loaded cached features", X.shape)
else:
    t0 = time.time()
    X, _ = F.build_feature_table(sample.clip_id.tolist(), C.CLEAN_DIR, with_logmel=False)
    X.to_csv(feat_path, index=False)
    print(f"Extracted {X.shape} in {(time.time() - t0) / 60:.1f} min")
df = sample.merge(X, on="clip_id")
fmiss = int(df[F.FEATURES].isna().sum().sum()); finf = int(np.isinf(df[F.FEATURES].values).sum())
print("Missing feature values:", fmiss, "| infinite:", finf)
if fmiss or finf:
    df[F.FEATURES] = df[F.FEATURES].replace([np.inf, -np.inf], np.nan)
    med = df.loc[df.split == "training", F.FEATURES].median()
    df[F.FEATURES] = df[F.FEATURES].fillna(med)      # training medians only (no leakage)
feat_qc = pd.DataFrame([{"missing_feature_values": fmiss, "infinite_values": finf,
                         "treatment": "median of training subset" if (fmiss or finf) else "none needed"}])
table(feat_qc, "T09_feature_qc")

In [ ]:
# Feature outliers: share of values beyond |z| > 4 in the training subset (kept; models use scaling / trees)
trn = df[df.split == "training"]
z = (trn[F.FEATURES] - trn[F.FEATURES].mean()) / trn[F.FEATURES].std()
out_feat = (z.abs() > 4).mean().sort_values(ascending=False) * 100
table(out_feat.rename("pct_beyond_4sd").round(2).reset_index().rename(columns={"index": "feature"}).head(10), "T10_feature_outliers_top10")

## 5. RQ1 EDA: which features separate human from synthetic speech?

In [ ]:
from src import evaluation as E, sample_size as S
pool = df[df.split.isin(["training", "validation"])]
res = E.feature_tests(pool, F.FEATURES)
rows = []
for f in res.feature.head(12):
    est, lo, hi = E.cluster_bootstrap(pool, lambda d, f=f: E.cohens_d(d.loc[d.label == 1, f].values, d.loc[d.label == 0, f].values), n_boot=300)
    rows.append({"feature": f, "ci_low": lo, "ci_high": hi})
top = res.head(12).merge(pd.DataFrame(rows), on="feature")
table(res.round(5), "T11_rq1_all_feature_tests")
table(top[["feature", "mean_human", "mean_synthetic", "cohens_d", "ci_low", "ci_high", "cliffs_delta", "welch_q_bh"]].round(4), "T12_rq1_top12_effects")
print("Features with |d| >= 0.3:", int((res.abs_d >= 0.3).sum()), "of 64 | |d| >= 0.8:", int((res.abs_d >= 0.8).sum()),
      "| significant after BH (q<.05):", int((res.welch_q_bh < 0.05).sum()))
fig, ax = plt.subplots(figsize=(7, 4.4))
t = top.iloc[::-1]
ax.errorbar(t.cohens_d, range(len(t)), xerr=[t.cohens_d - t.ci_low, t.ci_high - t.cohens_d], fmt="o", color=HUMAN, ecolor="grey", capsize=3)
ax.axvline(0, color="black", lw=0.8)
for v in (-0.8, -0.5, -0.2, 0.2, 0.5, 0.8): ax.axvline(v, color="grey", lw=0.5, ls=":")
ax.set_yticks(range(len(t))); ax.set_yticklabels(t.feature); ax.set_xlabel("Cohen's d (synthetic − human), 95% speaker-cluster bootstrap CI")
save(fig, "F04_rq1_top_effect_sizes")
top[["feature", "cohens_d", "ci_low", "ci_high", "cliffs_delta"]].round(3)

In [ ]:
top4 = res.feature.head(4).tolist()
fig, axes = plt.subplots(1, 4, figsize=(11, 3))
for ax, f in zip(axes, top4):
    data = [pool.loc[pool.label == 0, f], pool.loc[pool.label == 1, f]]
    vp = ax.violinplot(data, showmedians=True)
    for b, c in zip(vp["bodies"], [HUMAN, SYN]): b.set_facecolor(c); b.set_alpha(0.6)
    ax.set_xticks([1, 2]); ax.set_xticklabels(["human", "synthetic"]); ax.set_title(f, fontsize=10)
save(fig, "F05_rq1_top4_distributions")

In [ ]:
corr = trn[F.FEATURES].corr()
high = [(a, b, corr.loc[a, b]) for i, a in enumerate(F.FEATURES) for b in F.FEATURES[i + 1:] if abs(corr.loc[a, b]) > 0.8]
print("Feature pairs with |r| > 0.8:", len(high))
table(pd.DataFrame(high, columns=["feature_a", "feature_b", "r"]).round(3), "T13_high_correlation_pairs")
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ticks = [0, 20, 40, 50, 57, 60]; labels = ["MFCC means", "MFCC SDs", "Spectral shape", "Contrast", "Energy", "Prosody"]
ax.set_xticks(ticks); ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=9); ax.set_yticks(ticks); ax.set_yticklabels(labels, fontsize=9)
fig.colorbar(im, ax=ax, shrink=0.8, label="Pearson r")
save(fig, "F06_feature_correlation")

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
sc = StandardScaler().fit(trn[F.FEATURES])
pca = PCA(n_components=2, random_state=C.SEED).fit(sc.transform(trn[F.FEATURES]))
ev = pca.explained_variance_ratio_
vis = df[df.split.isin(["training", "test_unknown", "test_human"])]
vis = vis.sample(n=min(3000, len(vis)), random_state=C.SEED)
P = pca.transform(sc.transform(vis[F.FEATURES]))
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for name, colr in [("human", HUMAN), ("synthetic", SYN)]:
    m = (vis.label_name == name).values
    axes[0].scatter(P[m, 0], P[m, 1], s=4, alpha=0.4, c=colr, label=name)
axes[0].legend(frameon=False, markerscale=3); axes[0].set_xlabel(f"PC1 ({ev[0]:.0%})"); axes[0].set_ylabel(f"PC2 ({ev[1]:.0%})")
kn = vis.system_status.fillna("").map({"known": "known systems", "unseen": "unseen systems", "": "human"})
for name, colr in [("human", HUMAN), ("known systems", "#9DB4CC"), ("unseen systems", SYN)]:
    m = (kn == name).values
    axes[1].scatter(P[m, 0], P[m, 1], s=4, alpha=0.4, c=colr, label=name)
axes[1].legend(frameon=False, markerscale=3); axes[1].set_xlabel(f"PC1 ({ev[0]:.0%})")
save(fig, "F07_pca_by_class_and_system")
print("PCA explained variance PC1, PC2:", ev.round(3))

In [ ]:
best_f = res.feature.iloc[0]
syn_all = df[df.label == 1]
sys_stats = syn_all.groupby(["attack_system_id", "system_status"])[best_f].agg(["mean", "std", "count"]).reset_index()
human_mean = df.loc[df.label == 0, best_f].mean()
table(sys_stats.round(4), "T14_top_feature_by_system")
fig, ax = plt.subplots(figsize=(8, 3.4))
colors = [("#9DB4CC" if s == "known" else SYN) for s in sys_stats.system_status]
ax.bar(sys_stats.attack_system_id, sys_stats["mean"], color=colors, yerr=sys_stats["std"], capsize=2)
ax.axhline(human_mean, color=HUMAN, lw=1.5, ls="--", label="human mean")
ax.set_ylabel(best_f); ax.set_xlabel("Generation system (light = known, dark = unseen)"); ax.legend(frameon=False)
save(fig, "F08_top_feature_by_system")

## 6. Sample-size check with the estimated intra-class correlation

In [ ]:
icc = pd.Series({f: S.estimate_icc(pool[f].values, pool.speaker_id.values) for f in F.FEATURES})
rho_hat = float(np.clip(icc.median(), 0, None))
print(f"Median ICC across 64 features: {rho_hat:.3f} (IQR {icc.quantile(.25):.3f}-{icc.quantile(.75):.3f}); planning value 0.05")
tab_plan, n_plan = S.summary(0.05); tab_est, n_est = S.summary(rho_hat)
ss = tab_plan[["rq", "method", "minimum_N", "planned"]].rename(columns={"minimum_N": "min_N_rho_0.05"})
ss["min_N_rho_estimated"] = tab_est.minimum_N.values; ss["meets_with_estimated_rho"] = ss.planned >= ss.min_N_rho_estimated
table(ss, "T15_sample_size_with_estimated_icc")
json.dump({"rho_hat": rho_hat, "icc_q1": float(icc.quantile(.25)), "icc_q3": float(icc.quantile(.75)), "final_N_planned": n_plan, "final_N_estimated": n_est},
          open(OUT / "icc_summary.json", "w"), indent=1)
ss

## 7. Baseline models (RQ2 preliminary, RQ3 preliminary, RQ4 cost threshold on clean audio)

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.inspection import permutation_importance
from src import models as M
idx = {s: np.where(df.split == s)[0] for s in df.split.unique()}
tr, va = idx["training"], idx["validation"]
tk = np.r_[idx["test_human"], idx["test_known"]]; tu = np.r_[idx["test_human"], idx["test_unknown"]]
y = df.label.values; Xf = df[F.FEATURES]
rows, scores, fitted = [], {}, {}
dummy = DummyClassifier(strategy="prior").fit(Xf.iloc[tr], y[tr])
for name in ["LR", "SVM", "RF", "XGBoost"]:
    t0 = time.time()
    pipe, params, v_eer = M.tune_classical(name, Xf.iloc[tr], y[tr], Xf.iloc[va], y[va])
    secs = time.time() - t0
    thr = E.eer(y[va], pipe.predict_proba(Xf.iloc[va])[:, 1])[1]
    fitted[name] = (pipe, thr, params)
    for setname, r in [("validation", va), ("test_known", tk), ("test_unknown", tu)]:
        s = pipe.predict_proba(Xf.iloc[r])[:, 1]; scores[(name, setname)] = s
        rows.append({"model": name, "set": setname, "params": json.dumps(params), "tune_seconds": round(secs, 1),
                     **E.classification_report(y[r], s, thr)})
    print(f"{name}: val EER {v_eer:.4f}  ({secs:.0f}s)")
for setname, r in [("validation", va), ("test_known", tk), ("test_unknown", tu)]:
    s = dummy.predict_proba(Xf.iloc[r])[:, 1]
    rows.append({"model": "Majority baseline", "set": setname, "params": "prior", "tune_seconds": 0, **E.classification_report(y[r], s, 0.5)})
perf = table(pd.DataFrame(rows).round(4), "T16_baseline_performance")
perf.pivot(index="model", columns="set", values=["eer", "roc_auc", "f1", "recall_tpr", "fpr"]).round(3)

In [ ]:
# 95% speaker-cluster bootstrap CIs on Test-Known for EER and ROC-AUC
ci_rows = []
for name in ["LR", "SVM", "RF", "XGBoost"]:
    t = df.iloc[tk][["speaker_id", "label"]].reset_index(drop=True).assign(s=scores[(name, "test_known")])
    for metric, fn in {"eer": lambda d: E.eer(d.label.values, d.s.values)[0], "roc_auc": lambda d: E.roc_auc_score(d.label, d.s)}.items():
        est, lo, hi = E.cluster_bootstrap(t, fn, n_boot=500)
        ci_rows.append({"model": name, "metric": metric, "estimate": est, "ci_low": lo, "ci_high": hi})
table(pd.DataFrame(ci_rows).round(4), "T17_test_known_bootstrap_ci")

In [ ]:
from sklearn.metrics import roc_curve, confusion_matrix
best = perf[(perf.set == "validation") & (perf.model != "Majority baseline")].sort_values("eer").model.iloc[0]
print("Best model on validation EER:", best)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for name, ls in zip(["LR", "SVM", "RF", "XGBoost"], ["-", "--", "-.", ":"]):
    f_, t_, _ = roc_curve(y[tk], scores[(name, "test_known")]); axes[0].plot(f_, t_, ls, label=f"{name} (known)")
f_, t_, _ = roc_curve(y[tu], scores[(best, "test_unknown")]); axes[0].plot(f_, t_, color=SYN, lw=2, label=f"{best} (unseen)")
axes[0].plot([0, 1], [0, 1], color="grey", lw=0.8); axes[0].set_xlabel("False-positive rate"); axes[0].set_ylabel("True-positive rate"); axes[0].legend(frameon=False, fontsize=8)
pipe, thr, _ = fitted[best]
cm = confusion_matrix(y[tk], (scores[(best, "test_known")] >= thr).astype(int))
axes[1].imshow(cm, cmap="Blues")
for (i, j), v in np.ndenumerate(cm): axes[1].text(j, i, f"{v:,}", ha="center", va="center", color="black" if v < cm.max() / 2 else "white")
axes[1].set_xticks([0, 1]); axes[1].set_xticklabels(["pred. human", "pred. synthetic"]); axes[1].set_yticks([0, 1]); axes[1].set_yticklabels(["human", "synthetic"])
axes[1].set_title(f"{best}, Test-Known, threshold from validation", fontsize=10)
save(fig, "F09_roc_and_confusion")
table(pd.DataFrame(cm, index=["true_human", "true_synthetic"], columns=["pred_human", "pred_synthetic"]).reset_index(), "T18_confusion_matrix_best")

In [ ]:
# RQ3 preliminary: detection rate per generation system (best model, validation threshold)
syn_idx = np.r_[idx["test_known"], idx["test_unknown"]]
det = df.iloc[syn_idx][["attack_system_id", "attack_type", "system_status", "speaker_id"]].copy()
det["detected"] = (pipe.predict_proba(Xf.iloc[syn_idx])[:, 1] >= thr).astype(int)
per = []
for sid, g in det.groupby("attack_system_id"):
    est, lo, hi = E.cluster_bootstrap(g, lambda d: d.detected.mean(), n_boot=300)
    per.append({"system": sid, "type": g.attack_type.iloc[0], "status": g.system_status.iloc[0], "n": len(g), "detection_rate": est, "ci_low": lo, "ci_high": hi})
per = table(pd.DataFrame(per).sort_values("detection_rate").round(4), "T19_rq3_detection_by_system")
fig, ax = plt.subplots(figsize=(8, 3.6))
cols = ["#9DB4CC" if s == "known" else SYN for s in per.status]
ax.bar(per.system, per.detection_rate, color=cols, yerr=[per.detection_rate - per.ci_low, per.ci_high - per.detection_rate], capsize=2)
ax.set_ylim(0, 1.05); ax.set_ylabel("Detection rate (recall)"); ax.set_xlabel("Generation system (light = known, dark = unseen)")
save(fig, "F10_rq3_detection_by_system")
per

In [ ]:
# Feature importance (permutation, validation set, ROC-AUC drop)
pi = permutation_importance(pipe, Xf.iloc[va], y[va], scoring="roc_auc", n_repeats=5, random_state=C.SEED, n_jobs=-1)
imp = pd.DataFrame({"feature": F.FEATURES, "importance": pi.importances_mean, "sd": pi.importances_std}).sort_values("importance", ascending=False)
table(imp.round(5), "T20_permutation_importance")
fig, ax = plt.subplots(figsize=(7, 4)); t = imp.head(15).iloc[::-1]
ax.barh(t.feature, t.importance, xerr=t.sd, color=HUMAN); ax.set_xlabel(f"Drop in validation ROC-AUC when shuffled ({best})")
save(fig, "F11_permutation_importance")

In [ ]:
# RQ4 preliminary (clean audio only): cost-optimal thresholds chosen on validation, applied to Test-Known
s_val = pipe.predict_proba(Xf.iloc[va])[:, 1]; s_tk = scores[(best, "test_known")]
rows = []
for pi_ in C.PREVALENCES:
    opt = E.cost_optimal_threshold(y[va], s_val, pi_)
    r = E.classification_report(y[tk], s_tk, opt["threshold"])
    rows.append({"prevalence": pi_, "threshold": opt["threshold"], "test_tpr": r["recall_tpr"], "test_fpr": r["fpr"],
                 "precision_at_prevalence": E.precision_at_prevalence(r["recall_tpr"], r["fpr"], pi_),
                 "expected_cost_per_1000": E.expected_cost(r["recall_tpr"], r["fpr"], pi_), "cost_no_screening": 1000 * pi_ * C.COST_FN})
table(pd.DataFrame(rows).round(4), "T21_rq4_cost_thresholds_clean")
fig, ax = plt.subplots(figsize=(6.5, 3.6))
for pi_, ls in zip(C.PREVALENCES, ["-", "--"]):
    cc = E.cost_curve(y[va], s_val, pi_); ax.plot(cc.threshold.clip(0, 1), cc.expected_cost / 1000, ls, color=HUMAN, label=f"prevalence {pi_:.0%}")
ax.set_xlabel("Decision threshold on P(synthetic)"); ax.set_ylabel("Expected cost per 1,000 calls (₹ thousand)"); ax.legend(frameon=False)
save(fig, "F12_rq4_cost_curve")

## 8. Download everything as one zip

In [ ]:
import shutil
zip_path = shutil.make_archive(str(C.DATA_ROOT / "reports_interim"), "zip", OUT)
print("Saved:", zip_path, "| files:", len(list(OUT.iterdir())))
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)